In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 01: segment_overview
-- ══════════════════════════════════════════════════════
-- Full segment profile — the main table on the dashboard.

SELECT
  s.segment_label,
  COUNT(*)                                          AS fan_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_base,

  -- Revenue profile
  ROUND(AVG(c360.total_spend), 2)                  AS avg_total_spend,
  ROUND(AVG(c360.revenue_net), 2)                  AS avg_net_spend,
  ROUND(SUM(c360.revenue_net), 0)                  AS total_segment_revenue,

  -- Attendance profile
  ROUND(AVG(c360.games_attended), 1)               AS avg_games_attended,
  ROUND(AVG(c360.days_since_last), 0)              AS avg_days_since_last,

  -- Loyalty profile
  ROUND(AVG(c360.tenure_days), 0)                  AS avg_tenure_days,
  SUM(CASE WHEN c360.is_season_holder = TRUE THEN 1 ELSE 0 END) AS season_holders,

  -- Promo behavior
  ROUND(AVG(c360.promo_sensitivity), 3)            AS avg_promo_sensitivity,
  ROUND(AVG(c360.promo_rate), 3)                   AS avg_promo_rate,

  -- Churn risk
  ROUND(AVG(cs.churn_probability) * 100, 1)        AS avg_churn_prob_pct,
  SUM(CASE WHEN cs.risk_tier = 'High'
           THEN 1 ELSE 0 END)                       AS high_risk_fans,

  -- Jersey Night participation
  SUM(CASE WHEN c360.jersey_night_attendee = TRUE
           THEN 1 ELSE 0 END)                       AS jersey_night_attendees

FROM icebase.gold.ml_fan_segments       s
JOIN icebase.gold.customer_360          c360 ON s.customer_id = c360.customer_id
JOIN icebase.gold.ml_churn_scores       cs   ON s.customer_id = cs.customer_id
GROUP BY s.segment_label
ORDER BY total_segment_revenue DESC;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 02: segment_by_channel
-- ══════════════════════════════════════════════════════
-- Cross-tab: what segment did each acquisition channel produce?
-- Heatmap or stacked bar visualization.

SELECT
  d.acquisition_channel,
  s.segment_label,
  COUNT(*)                                          AS fan_count,
  ROUND(
    COUNT(*) * 100.0
    / SUM(COUNT(*)) OVER (PARTITION BY d.acquisition_channel),
    1
  )                                                 AS pct_within_channel,
  ROUND(AVG(cs.churn_probability) * 100, 1)        AS avg_churn_prob_pct
FROM icebase.silver.dim_customer        d
JOIN icebase.gold.ml_fan_segments       s  ON d.customer_id = s.customer_id
JOIN icebase.gold.ml_churn_scores       cs ON d.customer_id = cs.customer_id
GROUP BY d.acquisition_channel, s.segment_label
ORDER BY d.acquisition_channel, fan_count DESC;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 03: segment_spend_distribution
-- ══════════════════════════════════════════════════════
-- Spend percentiles per segment — for bar chart showing spread.

SELECT
  s.segment_label,
  ROUND(PERCENTILE(c360.revenue_net, 0.10), 2)     AS p10_spend,
  ROUND(PERCENTILE(c360.revenue_net, 0.25), 2)     AS p25_spend,
  ROUND(PERCENTILE(c360.revenue_net, 0.50), 2)     AS median_spend,
  ROUND(PERCENTILE(c360.revenue_net, 0.75), 2)     AS p75_spend,
  ROUND(PERCENTILE(c360.revenue_net, 0.90), 2)     AS p90_spend,
  ROUND(AVG(c360.revenue_net), 2)                  AS mean_spend,
  ROUND(MAX(c360.revenue_net), 2)                  AS max_spend
FROM icebase.gold.ml_fan_segments       s
JOIN icebase.gold.customer_360          c360 ON s.customer_id = c360.customer_id
GROUP BY s.segment_label
ORDER BY median_spend DESC;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 04: segment_game_attendance
-- ══════════════════════════════════════════════════════
-- Games attended distribution per segment.
-- Bar chart showing how often each segment shows up.

SELECT
  s.segment_label,
  CASE
    WHEN c360.games_attended = 0     THEN '0 games'
    WHEN c360.games_attended = 1     THEN '1 game'
    WHEN c360.games_attended <= 3    THEN '2–3 games'
    WHEN c360.games_attended <= 5    THEN '4–5 games'
    WHEN c360.games_attended <= 10   THEN '6–10 games'
    WHEN c360.games_attended <= 20   THEN '11–20 games'
    ELSE '21+ games'
  END                                               AS attendance_bucket,
  COUNT(*)                                          AS fan_count,
  CASE
    WHEN c360.games_attended = 0     THEN 0
    WHEN c360.games_attended = 1     THEN 1
    WHEN c360.games_attended <= 3    THEN 2
    WHEN c360.games_attended <= 5    THEN 3
    WHEN c360.games_attended <= 10   THEN 4
    WHEN c360.games_attended <= 20   THEN 5
    ELSE 6
  END                                               AS bucket_order
FROM icebase.gold.ml_fan_segments       s
JOIN icebase.gold.customer_360          c360 ON s.customer_id = c360.customer_id
GROUP BY s.segment_label, attendance_bucket, bucket_order
ORDER BY s.segment_label, bucket_order;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 05: ml_model_summary
-- ══════════════════════════════════════════════════════
-- Model metadata card — shows what's powering the dashboard.
-- Single-row display for a text/table widget in the header area.

SELECT
  'K-Means Fan Segmentation'                        AS segmentation_model,
  'icebase.gold.kmeans_fan_segmentation'            AS segmentation_registry,
  5                                                 AS n_segments,
  'XGBoost Churn Predictor'                         AS churn_model,
  'icebase.gold.xgboost_churn_predictor'            AS churn_registry,
  0.94                                              AS churn_model_auc,
  0.85                                              AS churn_model_accuracy,
  (SELECT MAX(scored_at) FROM icebase.gold.ml_churn_scores)
                                                    AS last_scored_at,
  (SELECT COUNT(DISTINCT segment_label)
   FROM icebase.gold.ml_fan_segments)               AS active_segments;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 06: segment_campaign_guide
-- ══════════════════════════════════════════════════════
-- Recommended marketing action per segment — table widget.
-- This is the "decision layer" that makes the dashboard actionable.

SELECT segment_label, fan_count, avg_churn_prob_pct,
       campaign_priority, recommended_channel,
       recommended_offer, campaign_goal
FROM (
  SELECT
    s.segment_label,
    COUNT(*)                                              AS fan_count,
    ROUND(AVG(cs.churn_probability) * 100, 1)            AS avg_churn_prob_pct,

    CASE s.segment_label
      WHEN 'Season Core'    THEN '5 — Maintain'
      WHEN 'High Value New' THEN '2 — Nurture'
      WHEN 'Casual Fan'     THEN '3 — Re-engage'
      WHEN 'Promo Hunter'   THEN '4 — Reduce promo dependency'
      WHEN 'Lapsed'         THEN '1 — Win back'
      WHEN 'Deeply Lapsed'  THEN '1 — Win back'
      ELSE '3 — Monitor'
    END                                                   AS campaign_priority,

    CASE s.segment_label
      WHEN 'Season Core'    THEN 'Direct mail + VIP event invite'
      WHEN 'High Value New' THEN 'Email welcome series + next-game offer'
      WHEN 'Casual Fan'     THEN 'SMS + social retargeting'
      WHEN 'Promo Hunter'   THEN 'Value bundle (not pure discount)'
      WHEN 'Lapsed'         THEN 'Email win-back + urgency offer'
      WHEN 'Deeply Lapsed'  THEN 'Heritage/nostalgia campaign'
      ELSE 'Email'
    END                                                   AS recommended_channel,

    CASE s.segment_label
      WHEN 'Season Core'    THEN 'Early renewal discount + exclusive access'
      WHEN 'High Value New' THEN '10% off next game'
      WHEN 'Casual Fan'     THEN 'Buy 2 get 1 free'
      WHEN 'Promo Hunter'   THEN 'Merch bundle (jersey + ticket)'
      WHEN 'Lapsed'         THEN '25% off return game'
      WHEN 'Deeply Lapsed'  THEN 'Free ticket + legacy fan recognition'
      ELSE 'Standard offer'
    END                                                   AS recommended_offer,

    CASE s.segment_label
      WHEN 'Season Core'    THEN 'Retain and upsell'
      WHEN 'High Value New' THEN 'Convert to repeat buyer'
      WHEN 'Casual Fan'     THEN 'Increase frequency'
      WHEN 'Promo Hunter'   THEN 'Shift to value perception'
      WHEN 'Lapsed'         THEN 'Reactivate within 30 days'
      WHEN 'Deeply Lapsed'  THEN 'Emotional reconnection'
      ELSE 'Maintain'
    END                                                   AS campaign_goal

  FROM icebase.gold.ml_fan_segments       s
  JOIN icebase.gold.ml_churn_scores       cs ON s.customer_id = cs.customer_id
  GROUP BY s.segment_label
)
ORDER BY campaign_priority;

In [0]:
-- ══════════════════════════════════════════════════════
-- DATASET 07: segment_over_time
-- ══════════════════════════════════════════════════════
-- Which segments did each acquisition phase produce?
-- Stacked bar chart — shows that the hot start produced many
-- fans who ended up as Lapsed or Casual (the bad story).

SELECT
  r.acquisition_phase,
  s.segment_label,
  COUNT(*)                                          AS fan_count,
  ROUND(
    COUNT(*) * 100.0
    / SUM(COUNT(*)) OVER (PARTITION BY r.acquisition_phase),
    1
  )                                                 AS pct_within_phase,
  CASE r.acquisition_phase
    WHEN 'hot_start'    THEN 1
    WHEN 'slump'        THEN 2
    WHEN 'late_push'    THEN 3
    WHEN 'bridge'       THEN 4
    WHEN 'jersey_night' THEN 5
    WHEN 'post_jersey'  THEN 6
    ELSE 7
  END                                               AS phase_order
FROM icebase.gold.retention_cohort      r
JOIN icebase.gold.ml_fan_segments       s ON r.customer_id = s.customer_id
WHERE r.acquisition_phase IS NOT NULL
GROUP BY r.acquisition_phase, s.segment_label
ORDER BY phase_order, fan_count DESC;